In [37]:
import yfinance as yf
import pandas as pd
import numpy as np

# Set print options for clean viewing in Jupyter
pd.set_option('display.max_columns', None)

In [38]:
# Define parameters
TICKER_SYMBOL = "NVDA"  # Feel free to change this to any ticker (e.g., "SPY", "MSFT")
HORIZON = "1y"
BB_WINDOW = 20
BB_STD = 2

# Fetch data using history() to get a clean DataFrame
ticker = yf.Ticker(TICKER_SYMBOL)
df = ticker.history(period=HORIZON)

# Calculate Bollinger Bands
df['SMA'] = df['Close'].rolling(window=BB_WINDOW).mean()
df['STD'] = df['Close'].rolling(window=BB_WINDOW).std()
df['Upper_Band'] = df['SMA'] + (df['STD'] * BB_STD)
df['Lower_Band'] = df['SMA'] - (df['STD'] * BB_STD)

# Drop rows with NaN values resulting from the rolling window
df.dropna(inplace=True)

print(col for col in df.columns)
df[['Close', 'Lower_Band']].head()

<generator object <genexpr> at 0x114c80110>


,Close,Lower_Band
Date,,
2025-07-10 00:00:00-04:00,163.881744,136.814848
2025-07-11 00:00:00-04:00,164.700638,137.516963
2025-07-14 00:00:00-04:00,163.851776,138.225509
2025-07-15 00:00:00-04:00,170.472931,139.073448
2025-07-16 00:00:00-04:00,171.142059,139.775998


In [39]:
# Trigger: Day's Low is less than or equal to the Lower Band
df['Triggered'] = df['Low'] <= df['Lower_Band']

# Shift Close prices backward by 1 to get "tomorrow's close" relative to today
df['Tomorrow_Close'] = df['Close'].shift(-1)

# Success condition: Tomorrow's close > Today's close
df['Is_Success'] = df['Tomorrow_Close'] > df['Close']

# Preview the instances where the strategy triggered
triggered_days = df[df['Triggered']]
triggered_days[['Close', 'Lower_Band', 'Tomorrow_Close', 'Is_Success']].head()

,Close,Lower_Band,Tomorrow_Close,Is_Success
Date,,,,
2025-08-20 00:00:00-04:00,175.166702,171.991160,174.747253,False
2025-08-22 00:00:00-04:00,177.753250,172.942555,179.570831,True
2025-08-29 00:00:00-04:00,173.948318,173.969815,170.552841,False
2025-09-02 00:00:00-04:00,170.552841,172.222205,170.393066,False
2025-09-03 00:00:00-04:00,170.393066,170.826211,171.431686,True


In [40]:
# Drop the last row to prevent look-ahead or null errors on the most recent day
backtest_df = df.iloc[:-1].copy()

# Calculate totals
total_triggers = backtest_df['Triggered'].sum()
total_successes = backtest_df[backtest_df['Triggered']]['Is_Success'].sum()

# Calculate Success Rate
if total_triggers > 0:
    success_rate = (total_successes / total_triggers) * 100
else:
    success_rate = 0.0

# Output results
print(f"=== Bollinger Band Backtest Results ({TICKER_SYMBOL}) ===")
print(f"Time Horizon: {HORIZON}")
print(f"Total times price touched/dropped below bottom band: {total_triggers}")
print(f"Total times price rebounded/closed higher next day: {total_successes}")
print(f"Strategy Success Rate: {success_rate:.2f}%")

=== Bollinger Band Backtest Results (NVDA) ===
Time Horizon: 1y
Total times price touched/dropped below bottom band: 24
Total times price rebounded/closed higher next day: 15
Strategy Success Rate: 62.50%


In [41]:
import plotly.graph_objects as go

# 1. Strip timezones for clean Plotly rendering
df_plot = df.copy()
if df_plot.index.tz is not None:
    df_plot.index = df_plot.index.tz_localize(None)

# 2. Isolate successful and failed triggers (excluding the last row)
plot_triggers = df_plot.iloc[:-1]
success_signals = plot_triggers[plot_triggers['Triggered'] & plot_triggers['Is_Success']]
fail_signals = plot_triggers[plot_triggers['Triggered'] & ~plot_triggers['Is_Success']]

# 3. Initialize Figure
fig = go.Figure()

# Add the Candlestick trace
fig.add_trace(go.Candlestick(
    x=df_plot.index,
    open=df_plot['Open'],
    high=df_plot['High'],
    low=df_plot['Low'],
    close=df_plot['Close'],
    name='OHLC',
    increasing_line_color='green', 
    decreasing_line_color='red'
))

# Add Bollinger Bands & SMA Lines
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['Upper_Band'], name='Upper Band', line=dict(color='gray', width=1, dash='dash'), opacity=0.5))
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['Lower_Band'], name='Lower Band', line=dict(color='gray', width=1, dash='dash'), opacity=0.5, fill='tonexty', fillcolor='rgba(128,128,128,0.05)'))
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['SMA'], name='20 SMA', line=dict(color='orange', width=1, dash='dot')))

# Dynamic visual padding to drop the triangles slightly below the candle wicks
padding = df_plot['Close'].mean() * 0.006  

# Add Entry Buy Signals (Success)
fig.add_trace(go.Scatter(
    x=success_signals.index,
    y=success_signals['Low'] - padding,
    mode='markers',
    marker=dict(symbol='triangle-up', size=11, color='green'),
    name='Trigger: Success'
))

# Add Entry Buy Signals (Fail)
fig.add_trace(go.Scatter(
    x=fail_signals.index,
    y=fail_signals['Low'] - padding,
    mode='markers',
    marker=dict(symbol='triangle-down', size=11, color='red'),
    name='Trigger: Fail'
))

# 4. Layout Configurations (Enables Zoom, Pan, and Range Slider)
fig.update_layout(
    title=f"{TICKER_SYMBOL} Bollinger Band Backtest Visual ({HORIZON})",
    yaxis_title='Price ($)',
    xaxis_title='Date',
    template='plotly_white',
    height=700,
    hovermode='x unified',  # Shows all data points on hover cleanly
    xaxis_rangeslider_visible=True  # Bottom timeline slider for macro zooming
)

# Render the interactive chart inside the notebook
fig.show()